In [1]:
! pip install openai

In [2]:
from openai import OpenAI
import json

In [3]:
# Connect to your local Ollama server
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [4]:
# --- Local tools ---
def calculator(expression: str) -> str:
    """Safe math calculator."""
    try:
        allowed_chars = "0123456789+-*/(). "
        if not all(c in allowed_chars for c in expression):
            return "Error: Invalid characters"
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

def local_knowledge(query: str) -> str:
    """Pretend local knowledge base."""
    data = {
        "capital of france": "Paris",
        "python creator": "Guido van Rossum",
        "largest ocean": "Pacific Ocean",
    }
    return data.get(query.lower(), "No results found in local database.")

TOOLS = {
    "calculator": calculator,
    "local_search": local_knowledge,
}

In [5]:
# --- Agent system prompt ---
SYSTEM_PROMPT = """
You are a local AI agent that can use tools.
You have no internet access.
Use these tools to answer questions step by step.

Tools:
- calculator(expression)
- local_search(query)

Always respond in JSON:
{"action": "tool_name", "input": "..."}
or, when done:
{"action": "final_answer", "input": "..."}
"""

In [6]:
def run_local_agent(query: str, max_steps=5):
    print(f"🧠 User: {query}")
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": query},
    ]

    for step in range(max_steps):
        # If there's a tool result from last step, add it
        if step > 0 and 'result' in locals():
            messages.append({
                "role": "user",
                "content": "The tool returned: " + result + 
                        "\nWhat should you do next? If you have enough information, return the final answer."
            })

        # Ask local model
        response = client.chat.completions.create(
            model="llama3",  # or any local model you pulled
            messages=messages,
        )

        choice = response.choices[0].message
        content = choice.content or ""  # safely handle None
        if not content.strip():
            print("⚠️ Empty response from model.")
            continue

        content = content.strip()
        print(f"Step {step+1} → {content}\n")

        # Parse tool or final answer
        try:
            action = json.loads(content)
            name = action["action"]
            arg = action["input"]
        except Exception:
            print("⚠️ Invalid JSON from model.")
            break

        if name == "final_answer":
            print(f"✅ Final Answer: {arg}")
            break

        elif name in TOOLS:
            result = TOOLS[name](arg)
            print(f"🧰 {name}('{arg}') → {result}\n")
            messages.append({"role": "assistant", "content": content})
            messages.append({"role": "assistant", "content": f"Tool result: {result}"})
        else:
            print(f"⚠️ Unknown tool '{name}'")
            break
    else:
        print("⏹️ Reached max reasoning steps.")

In [7]:
run_local_agent("What is (15 + 5) * 3?")

🧠 User: What is (15 + 5) * 3?
Step 1 → {"action": "calculator", "input": "(15 + 5) * 3"}

🧰 calculator('(15 + 5) * 3') → 60

Step 2 → {"action": "final_answer", "input": "60"}

✅ Final Answer: 60
